## **Speed dating — lecture des données**

Chaque graphique répond à une question simple : 
- à quoi ressemble une soirée de rencontres, 
- et qu’est-ce qui accompagne un match ou un passage à côté.


### **Fil de l’analyse**

D’abord les ordres de grandeur et les répartitions, puis le détail des notes et des profils, enfin les **croisements** pour voir ce qui accompagne un match ou un passage à côté — le tout en graphiques interactifs à parcourir ici.


### **Mise en route**

On charge les librairies et le jeu de données déjà nettoyé : une ligne correspond à une rencontre entre deux personnes lors d’une soirée.


In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from pathlib import Path
 
pio.templates.default = "plotly_white"
 
# ── Palette & constantes ────────────
C = {
    "match":   "#1D9E75", "no_match": "#E24B4A",
    "male":    "#378ADD", "female":   "#D4537E",
    "purple":  "#7F77DD", "amber":    "#EF9F27",
    "teal":    "#0FA98E", "coral":    "#D85A30",
    "neutral": "#888780", "blue":     "#3B82C4",
}
COLORS8 = ["#7F77DD","#1D9E75","#378ADD","#EF9F27","#D85A30","#888780","#D4537E","#E24B4A"]
 
SCORE_SELF  = ["attr","sinc","intel","fun","amb","shar","like","prob"]
SCORE_OTHER = ["attr_o","sinc_o","intel_o","fun_o","amb_o","shar_o","like_o","prob_o"]
PREF_COLS   = ["attr1_1","sinc1_1","intel1_1","fun1_1","amb1_1","shar1_1"]
SELF_COLS   = ["attr3_1","sinc3_1","intel3_1","fun3_1","amb3_1"]
 
ATTR_FR = {
    "attr":"Attractivité", "sinc":"Sincérité", "intel":"Intelligence",
    "fun":"Fun", "amb":"Ambition", "shar":"Intérêts communs",
    "like":"Appréciation", "prob":"Proba. match perçue",
    "attr1_1":"Attractivité", "sinc1_1":"Sincérité", "intel1_1":"Intelligence",
    "fun1_1":"Fun", "amb1_1":"Ambition", "shar1_1":"Intérêts communs",
    "attr3_1":"Attractivité", "sinc3_1":"Sincérité", "intel3_1":"Intelligence",
    "fun3_1":"Fun", "amb3_1":"Ambition",
    "attr_o":"Attractivité", "sinc_o":"Sincérité", "intel_o":"Intelligence",
    "fun_o":"Fun", "amb_o":"Ambition", "shar_o":"Intérêts communs",
    "like_o":"Appréciation", "prob_o":"Proba. match perçue",
}

ATTR6 = ["attr","sinc","intel","fun","amb","shar"]
ATTR6_FR = ["Attractivité","Sincérité","Intelligence","Fun","Ambition","Intérêts communs"]
 
FIELD_MAP = {
    1:"Droit", 2:"Maths", 3:"Sciences sociales", 4:"Médecine",
    5:"Ingénierie", 6:"Lettres/Journalisme", 7:"Histoire/Philo",
    8:"Business/Finance", 9:"Éducation", 10:"Bio/Chimie/Physique",
    11:"Travail social", 12:"Undergrad", 13:"Sciences politiques",
    14:"Cinéma", 15:"Beaux-Arts", 16:"Langues", 17:"Architecture", 18:"Autre"
}
RACE_MAP = {1:"Noir", 2:"Blanc", 3:"Latino", 4:"Asiatique", 5:"Amérindien", 6:"Autre"}
GOOUT_MAP = {1:"Plusieurs/sem", 2:"2x/sem", 3:"1x/sem",
             4:"2x/mois", 5:"1x/mois", 6:"Qqfois/an", 7:"Rarement"}

PROJECT_ROOT = Path('../')
INPUT_DATA_PATH = PROJECT_ROOT / 'outputs' / 'data' / 'df_speed_dating_cleaned.csv'
print(INPUT_DATA_PATH)


../outputs/data/df_speed_dating_cleaned.csv


In [2]:
# =============================================================================
# CHARGEMENT
# =============================================================================
df = pd.read_csv(INPUT_DATA_PATH)
print(f"\nDataset chargé : {df.shape[0]:,} lignes x {df.shape[1]} colonnes")


Dataset chargé : 8,283 lignes x 49 colonnes


### **1. Statistiques descriptives — les ordres de grandeur**

Quelques moyennes et répartitions sur les indicateurs centraux : le **match**, l’intention de revoir la personne (**dec**), et les notes typiques.


In [3]:
# =============================================================================
# Chiffres clés (moyennes, taux)
# =============================================================================
summary_cols = [c for c in ["match", "dec", "attr", "fun", "like", "prob", "int_corr", "samerace", "age", "go_out"] if c in df.columns]
print(df[summary_cols].describe().T)
print(f"\nTaux de match: {df['match'].mean()*100:.1f}%")
if "dec" in df.columns:
    print(f"Taux dec=Oui: {df['dec'].mean()*100:.1f}%")
if "gender_label" in df.columns:
    print("\nTaux de match par genre:")
    print((df.groupby("gender_label")["match"].mean() * 100).round(2))


           count       mean       std    min    25%    50%    75%    max
match     8283.0   0.164433  0.370691   0.00   0.00   0.00   0.00   1.00
dec       8283.0   0.421224  0.493785   0.00   0.00   0.00   1.00   1.00
attr      8283.0   6.193094  1.931032   0.00   5.00   6.00   8.00  10.00
fun       8283.0   6.412411  1.913320   0.00   5.00   7.00   8.00  10.00
like      8283.0   6.130774  1.816015   0.00   5.00   6.00   7.00  10.00
prob      8283.0   5.207835  2.093503   0.00   4.00   5.00   7.00  10.00
int_corr  8283.0   0.195180  0.302263  -0.83  -0.01   0.21   0.43   0.91
samerace  8283.0   0.398165  0.489549   0.00   0.00   0.00   1.00   1.00
age       8283.0  26.358928  3.566763  18.00  24.00  26.00  28.00  55.00
go_out    8283.0   2.157189  1.105918   1.00   1.00   2.00   3.00   7.00

Taux de match: 16.4%
Taux dec=Oui: 42.1%

Taux de match par genre:
gender_label
Femme    16.51
Homme    16.38
Name: match, dtype: float64


### **2. Analyse univariée — une dimension à la fois**

On isole chaque famille d’informations : part des matchs, notes de la soirée, priorités déclarées avant l’événement, profil démographique, puis quelques indicateurs synthétiques (écarts d’âge, alignement des intérêts, etc.).


In [4]:
# =============================================================================
# LE MATCH COMME FIL ROUGE
# =============================================================================
print("\n" + "─" * 65)
print("Match : part des rencontres et décisions")
print("─" * 65)
 
match_rate   = df["match"].mean() * 100
match_by_g   = df.groupby("gender_label")["match"].mean() * 100
match_by_w   = df.groupby("wave")["match"].mean() * 100
dec_rate_f   = df[df["gender"]==0]["dec"].mean() * 100
dec_rate_m   = df[df["gender"]==1]["dec"].mean() * 100
 
print(f"\nTaux de match global          : {match_rate:.1f}%")
print(f"Taux de match — Femmes        : {match_by_g['Femme']:.1f}%")
print(f"Taux de match — Hommes        : {match_by_g['Homme']:.1f}%")
print(f"> Quasi-identiques : symétrie parfaite du dataset!")
print(f"\nTaux dec=Oui — Femmes         : {dec_rate_f:.1f}%")
print(f"Taux dec=Oui — Hommes         : {dec_rate_m:.1f}%")
print(f"> Les hommes disent 'Oui' bien plus souvent ({dec_rate_m:.0f}% vs {dec_rate_f:.0f}%)")
print(f"> Le match est donc limité par la sélectivité féminine")
print(f"\nTaux de match par vague :")
print(f"Min : {match_by_w.min():.1f}%  |  Max : {match_by_w.max():.1f}%  |  Écart : {match_by_w.max()-match_by_w.min():.1f}pp")
 
# Figure 1a : Pie match
fig_u1a = go.Figure(go.Pie(
    labels=["Pas de match", "Match"],
    values=df["match"].value_counts().sort_index().values,
    marker_colors=[C["no_match"], C["match"]],
    hole=0.52, textinfo="label+percent",
))
fig_u1a.update_layout(
    title="Part des rencontres qui aboutissent à un match",
    height=360, title_font_size=15)
fig_u1a.show()
 
# Figure 1b : Décision Oui + Match par vague
fig_u1b = make_subplots(rows=1, cols=2,
    subplot_titles=["Taux décision 'Oui' par genre", "Taux de match par vague"])
fig_u1b.add_trace(go.Bar(
    x=["Femme", "Homme"], y=[dec_rate_f, dec_rate_m],
    marker_color=[C["female"], C["male"]],
    text=[f"{dec_rate_f:.1f}%", f"{dec_rate_m:.1f}%"],
    textposition="outside", cliponaxis=False, showlegend=False, width=0.5,
), row=1, col=1)
fig_u1b.add_trace(go.Scatter(
    x=match_by_w.index, y=match_by_w.values,
    mode="lines+markers",
    marker=dict(color=C["teal"], size=8),
    line=dict(color=C["teal"], width=2),
    showlegend=False,
), row=1, col=2)
fig_u1b.add_hline(y=match_rate, line_dash="dot", line_color=C["neutral"],
    annotation_text=f"Moy. {match_rate:.1f}%", row=1, col=2)
fig_u1b.update_layout(
    title="Décision Oui par genre & taux de match par vague",
    height=380, plot_bgcolor="white", title_font_size=15)
fig_u1b.update_yaxes(title_text="% Oui", row=1, col=1)
fig_u1b.update_yaxes(title_text="% match", row=1, col=2)
fig_u1b.update_xaxes(title_text="Vague", row=1, col=2)
fig_u1b.show()


─────────────────────────────────────────────────────────────────
Match : part des rencontres et décisions
─────────────────────────────────────────────────────────────────

Taux de match global          : 16.4%
Taux de match — Femmes        : 16.5%
Taux de match — Hommes        : 16.4%
> Quasi-identiques : symétrie parfaite du dataset!

Taux dec=Oui — Femmes         : 36.8%
Taux dec=Oui — Hommes         : 47.4%
> Les hommes disent 'Oui' bien plus souvent (47% vs 37%)
> Le match est donc limité par la sélectivité féminine

Taux de match par vague :
Min : 8.3%  |  Max : 31.0%  |  Écart : 22.7pp


In [5]:
# =============================================================================
# NOTES DONNÉES À L’AUTRE PERSONNE
# =============================================================================
print("\n" + "─" * 65)
print("Notes sur l’autre personne (échelle 0 à 10)")
print("─" * 65)
 
score_stats = df[SCORE_SELF].agg(["mean","median","std","skew"]).T.round(2)
score_stats.index = [ATTR_FR.get(c, c) for c in score_stats.index]
print("\nSynthèse des notes (échelle 0 à 10) :")
print(score_stats.to_string())
print("\nObservations :")
print("- Sincérité (7.17) et Intelligence (7.38) : notes systématiquement hautes")
print("- Attractivité (6.20) et Fun (6.41) : plus discriminantes")
print("- Intérêts communs (5.54) : note la plus basse, forte variance")
print("- Toutes les distributions sont légèrement asymétriques à gauche (skew < 0)")
print("-> Les gens évitent les notes très basses (effet politesse)")
 
# Histogrammes 8 attributs
fig_u2a = make_subplots(
    rows=2, cols=4,
    subplot_titles=[ATTR_FR.get(c, c) for c in SCORE_SELF],
    vertical_spacing=0.16, horizontal_spacing=0.07,
)
for i, col in enumerate(SCORE_SELF):
    r, c = divmod(i, 4)
    mean_v = df[col].mean()
    fig_u2a.add_trace(go.Histogram(
        x=df[col], nbinsx=20,
        marker_color=COLORS8[i], opacity=0.85, showlegend=False,
    ), row=r+1, col=c+1)
    fig_u2a.add_vline(
        x=mean_v, line_dash="dash", line_color="black", line_width=1.5,
        annotation_text=f"μ={mean_v:.1f}", annotation_font_size=10,
        row=r+1, col=c+1
    )
fig_u2a.update_layout(
    title="Huit dimensions notées lors de la rencontre (0 à 10)",
    height=520, plot_bgcolor="white", title_font_size=15,
    margin=dict(t=95, b=75, l=75, r=45),
)
fig_u2a.show()
 
# Comparaison scores donnés vs reçus (radar)
avg_given    = df[SCORE_SELF[:6]].mean().values
avg_received = df[SCORE_OTHER[:6]].mean().values
attrs_radar  = [ATTR_FR.get(c, c) for c in SCORE_SELF[:6]]
 
fig_u2b = go.Figure()
fig_u2b.add_trace(go.Scatterpolar(
    r=list(avg_given) + [avg_given[0]],
    theta=attrs_radar + [attrs_radar[0]],
    fill="toself", name="Scores donnés (sujet -> partenaire)",
    line_color=C["purple"], fillcolor=C["purple"], opacity=0.25,
))
fig_u2b.add_trace(go.Scatterpolar(
    r=list(avg_received) + [avg_received[0]],
    theta=attrs_radar + [attrs_radar[0]],
    fill="toself", name="Scores reçus (partenaire -> sujet)",
    line_color=C["teal"], fillcolor=C["teal"], opacity=0.25,
))
fig_u2b.update_layout(
    polar=dict(radialaxis=dict(range=[4, 8])),
    title="Scores donnés vs reçus (moyenne globale par attribut)",
    height=460, title_font_size=15,
    margin=dict(t=95, b=75, l=75, r=45),
)
fig_u2b.show()
 
print(f"\n  Comparaison scores donnés vs reçus :")
for s, o in zip(SCORE_SELF[:6], SCORE_OTHER[:6]):
    diff = df[s].mean() - df[o].mean()
    direction = "-> sujet plus généreux" if diff > 0.1 else "-> partenaire plus généreux" if diff < -0.1 else "-> symétrique"
    print(f"  {ATTR_FR.get(s, s):<20}  donné={df[s].mean():.2f}  reçu={df[o].mean():.2f}  diff={diff:+.2f}  {direction}")


─────────────────────────────────────────────────────────────────
Notes sur l’autre personne (échelle 0 à 10)
─────────────────────────────────────────────────────────────────

Synthèse des notes (échelle 0 à 10) :
                     mean  median   std  skew
Attractivité         6.19     6.0  1.93 -0.33
Sincérité            7.18     7.0  1.72 -0.65
Intelligence         7.36     7.0  1.52 -0.53
Fun                  6.41     7.0  1.91 -0.48
Ambition             6.79     7.0  1.73 -0.43
Intérêts communs     5.49     6.0  2.02 -0.23
Appréciation         6.13     6.0  1.82 -0.45
Proba. match perçue  5.21     5.0  2.09 -0.13

Observations :
- Sincérité (7.17) et Intelligence (7.38) : notes systématiquement hautes
- Attractivité (6.20) et Fun (6.41) : plus discriminantes
- Intérêts communs (5.54) : note la plus basse, forte variance
- Toutes les distributions sont légèrement asymétriques à gauche (skew < 0)
-> Les gens évitent les notes très basses (effet politesse)



  Comparaison scores donnés vs reçus :
  Attractivité          donné=6.19  reçu=6.19  diff=+0.01  -> symétrique
  Sincérité             donné=7.18  reçu=7.18  diff=-0.01  -> symétrique
  Intelligence          donné=7.36  reçu=7.37  diff=-0.02  -> symétrique
  Fun                   donné=6.41  reçu=6.41  diff=+0.00  -> symétrique
  Ambition              donné=6.79  reçu=6.80  diff=-0.01  -> symétrique
  Intérêts communs      donné=5.49  reçu=5.49  diff=+0.00  -> symétrique


In [6]:
# =============================================================================
# PRIORITÉS DÉCLARÉES AVANT LA SOIRÉE
# =============================================================================
print("\n" + "─" * 65)
print("Ce que les participants disent valoriser avant l’événement")
print("─" * 65)
 
pref_gender = df.groupby("gender_label")[PREF_COLS].mean().T.reset_index()
pref_gender.columns = ["col","Femme","Homme"]
pref_gender["Attribut"] = pref_gender["col"].map(ATTR_FR)
pref_gender = pref_gender.sort_values("Femme", ascending=True)
 
print("\nImportance déclarée moyenne (0-10) :")
print(f"{'Attribut':<22} {'Femme':>8} {'Homme':>8} {'Écart':>8}")
print("  " + "─" * 50)
for _, row in pref_gender.sort_values("Femme", ascending=False).iterrows():
    diff = row["Homme"] - row["Femme"]
    flag = "H" if diff > 0.1 else "F" if diff < -0.1 else "="
    print(f"{row['Attribut']:<22} {row['Femme']:>8.2f} {row['Homme']:>8.2f} {diff:>+8.2f}  {flag}")
 
print("\n> Attractivité : attribut le plus important ET la plus grande différence H/F")
print("> Ambition & Intérêts communs : attributs les moins valorisés")
 
fig_u3 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "Importance déclarée par genre (moyenne, 0-10)",
        "Distribution par attribut"
    ],
    horizontal_spacing=0.12,
)
 
for genre, clr in [("Femme", C["female"]), ("Homme", C["male"])]:
    fig_u3.add_trace(go.Bar(
        y=pref_gender["Attribut"],
        x=pref_gender[genre],
        name=genre, orientation="h",
        marker_color=clr, opacity=0.85,
    ), row=1, col=1)
 
pref_short = {"attr1_1":"Attr","sinc1_1":"Sinc","intel1_1":"Intel",
              "fun1_1":"Fun","amb1_1":"Amb","shar1_1":"Shar"}
for col, clr in zip(PREF_COLS, COLORS8):
    fig_u3.add_trace(go.Box(
        y=df[col], name=pref_short.get(col, col),
        marker_color=clr, boxmean=True,
         showlegend=False, opacity=0.8,
    ), row=1, col=2)
 
fig_u3.update_layout(
    title="Ce que les participants disent valoriser avant la soirée",
    height=440, plot_bgcolor="white", title_font_size=15,
    margin=dict(t=95, b=75, l=75, r=45),
    barmode="group", legend_title="Genre",
)
fig_u3.update_xaxes(title_text="Score moyen (0-10)", row=1, col=1)
fig_u3.update_yaxes(title_text="Score (0-10)", row=1, col=2)
fig_u3.show()


─────────────────────────────────────────────────────────────────
Ce que les participants disent valoriser avant l’événement
─────────────────────────────────────────────────────────────────

Importance déclarée moyenne (0-10) :
Attribut                  Femme    Homme    Écart
  ──────────────────────────────────────────────────
Intelligence               2.10     1.95    -0.15  F
Sincérité                  1.83     1.65    -0.18  F
Attractivité               1.81     2.69    +0.88  H
Fun                        1.71     1.78    +0.06  =
Ambition                   1.29     0.86    -0.43  F
Intérêts communs           1.27     1.10    -0.17  F

> Attractivité : attribut le plus important ET la plus grande différence H/F
> Ambition & Intérêts communs : attributs les moins valorisés


In [7]:
# =============================================================================
# AUTO-ÉVALUATION T1
# =============================================================================
print("\n" + "─" * 65)
print("AUTO-ÉVALUATION T1 (comment je me vois)")
print("─" * 65)
 
self_stats = df[SELF_COLS].agg(["mean","median","std"]).T.round(2)
self_stats.index = [ATTR_FR.get(c, c) for c in self_stats.index]
print("\nAuto-évaluation moyenne (0-10) :")
print(self_stats.to_string())
print("\n> Intelligence (8.40) et Sincérité (8.29) : biais de sur-évaluation fort")
print("> Les gens se notent bien au-dessus de la moyenne des scores reçus")
print(f"=> Score attr reçu moyen : {df['attr_o'].mean():.2f}  vs auto-éval attr : {df['attr3_1'].mean():.2f}")
 
self_gender = df.groupby("gender_label")[SELF_COLS].mean().T.reset_index()
self_gender.columns = ["col","Femme","Homme"]
self_gender["Attribut"] = self_gender["col"].map(ATTR_FR)
 
fig_u4 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "Auto-évaluation par genre",
        "Moi vs Comment les autres me voient (attr)"
    ],
    horizontal_spacing=0.14,
)
 
for genre, clr in [("Femme", C["female"]), ("Homme", C["male"])]:
    fig_u4.add_trace(go.Bar(
        x=self_gender["Attribut"], y=self_gender[genre],
        name=genre, marker_color=clr, opacity=0.85,
    ), row=1, col=1)
 
# Comparaison auto-eval attr vs score reçu attr_o
for genre, clr, col_self, col_other in [
    ("Femme", C["female"], df[df["gender"]==0], df[df["gender"]==0]),
    ("Homme", C["male"],   df[df["gender"]==1], df[df["gender"]==1]),
]:
    sub = df[df["gender_label"] == genre]
    fig_u4.add_trace(go.Box(
        y=sub["attr3_1"], name=f"{genre} — auto-éval",
        marker_color=clr, opacity=0.75,
        boxmean=True, showlegend=False,
    ), row=1, col=2)
    fig_u4.add_trace(go.Box(
        y=sub["attr_o"], name=f"{genre} — score reçu",
        marker_color=clr, opacity=0.35,
        boxmean=True, showlegend=False,
    ), row=1, col=2)
 
fig_u4.update_layout(
    title="Comment chacun se voit avant la rencontre",
    height=420, plot_bgcolor="white", title_font_size=15,
    margin=dict(t=95, b=75, l=75, r=45),
    barmode="group", legend_title="Genre",
)
fig_u4.update_yaxes(title_text="Score (0-10)", row=1, col=1)
fig_u4.update_yaxes(title_text="Score attractivité (0-10)", row=1, col=2)
fig_u4.show()



─────────────────────────────────────────────────────────────────
AUTO-ÉVALUATION T1 (comment je me vois)
─────────────────────────────────────────────────────────────────

Auto-évaluation moyenne (0-10) :
              mean  median   std
Attractivité  7.08     7.0  1.39
Sincérité     8.29     8.0  1.41
Intelligence  8.41     8.0  1.07
Fun           7.70     8.0  1.56
Ambition      7.58     8.0  1.78

> Intelligence (8.40) et Sincérité (8.29) : biais de sur-évaluation fort
> Les gens se notent bien au-dessus de la moyenne des scores reçus
=> Score attr reçu moyen : 6.19  vs auto-éval attr : 7.08


In [8]:
# =============================================================================
# DÉMOGRAPHIE
# =============================================================================
print("\n" + "─" * 65)
print("DÉMOGRAPHIE")
print("─" * 65)
 
# Participants uniques seulement
df_part = df.drop_duplicates("iid").copy()
 
print(f"\nParticipants uniques : {len(df_part)}")
print(f"Âge moyen : {df_part['age'].mean():.1f} ans  |  Médiane : {df_part['age'].median():.0f} ans")
print(f"Domaine #1 : Business/Finance ({(df_part['field_cd']==8).sum()} participants)")
print(f"Race majoritaire : Blanc ({(df_part['race']==2).sum()} participants)")
print(f"Go_out 1-2x/sem : {((df_part['go_out']<=2)).sum()} participants ({((df_part['go_out']<=2)).mean()*100:.0f}%)")
 
fig_u5 = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Distribution de l'âge",
        "Domaine d'études (top 8)",
        "Race / Ethnicité",
        "Fréquence de sortie (go_out)"
    ],
    vertical_spacing=0.18, horizontal_spacing=0.12,
)
 
# Âge
fig_u5.add_trace(go.Histogram(
    x=df_part["age"], nbinsx=25,
    marker_color=C["purple"], opacity=0.85, showlegend=False,
), row=1, col=1)
fig_u5.add_vline(x=df_part["age"].mean(), line_dash="dash", line_color="black",
    annotation_text=f"μ={df_part['age'].mean():.1f}", row=1, col=1)
 
# Domaine (top 8)
field_cnt = df_part["field_cd"].value_counts().head(8).reset_index()
field_cnt.columns = ["code","count"]
field_cnt["label"] = field_cnt["code"].map(FIELD_MAP).fillna("Autre")
field_cnt = field_cnt.sort_values("count", ascending=True)
fig_u5.add_trace(go.Bar(
    y=field_cnt["label"], x=field_cnt["count"],
    orientation="h", marker_color=C["teal"],
    showlegend=False,
), row=1, col=2)
 
# Race
race_cnt = df_part["race"].value_counts().reset_index()
race_cnt.columns = ["code","count"]
race_cnt["label"] = race_cnt["code"].map(RACE_MAP).fillna("Autre")
fig_u5.add_trace(go.Bar(
    x=race_cnt["label"], y=race_cnt["count"],
    marker_color=C["amber"], showlegend=False,
    text=race_cnt["count"], textposition="outside", cliponaxis=False,
), row=2, col=1)
 
# Go_out
goout_cnt = df_part["go_out"].value_counts().sort_index().reset_index()
goout_cnt.columns = ["code","count"]
goout_cnt["label"] = goout_cnt["code"].map(GOOUT_MAP).fillna("?")
fig_u5.add_trace(go.Bar(
    x=goout_cnt["label"], y=goout_cnt["count"],
    marker_color=C["coral"], showlegend=False,
    text=goout_cnt["count"], textposition="outside", cliponaxis=False,
), row=2, col=2)
 
fig_u5.update_layout(
    title="Âge, filière, origine et habitudes de sortie (une ligne par participant)",
    height=580, plot_bgcolor="white", title_font_size=15,
    margin=dict(t=95, b=75, l=75, r=45),
)
fig_u5.update_xaxes(title_text="Âge", row=1, col=1)
fig_u5.update_xaxes(title_text="Nb participants", row=1, col=2)
fig_u5.update_xaxes(tickangle=-30, row=2, col=1)
fig_u5.update_xaxes(tickangle=-30, row=2, col=2)
fig_u5.show()


─────────────────────────────────────────────────────────────────
DÉMOGRAPHIE
─────────────────────────────────────────────────────────────────

Participants uniques : 543
Âge moyen : 26.4 ans  |  Médiane : 26 ans
Domaine #1 : Business/Finance (129 participants)
Race majoritaire : Blanc (302 participants)
Go_out 1-2x/sem : 368 participants (68%)


In [9]:
# =============================================================================
# INDICATEURS COMPOSÉS
# =============================================================================
print("\n" + "─" * 65)
print("Âge relatif, attentes vs réalité, signaux agrégés")
print("─" * 65)

df["age_diff"] = (df["age"] - df["age_o"]).abs()
df["attr_expectation_gap"] = (df["attr1_1"] - df["attr_o"]).round(2)
df["attr_received_avg"] = df.groupby("iid")["attr_o"].transform("mean").round(2)
df["score_received_avg"] = df[SCORE_OTHER].mean(axis=1).round(2)
df["score_given_avg"] = df[SCORE_SELF].mean(axis=1).round(2)

print(f"\nage_diff (écart d'âge absolu) :")
print(f"Moyenne : {df['age_diff'].mean():.2f} ans  |  Médiane : {df['age_diff'].median():.1f} ans")
print(f"75% des paires ont moins de {df['age_diff'].quantile(0.75):.0f} ans d'écart")
 
print(f"\nint_corr (corrélation des intérêts communs) :")
print(f"Moyenne : {df['int_corr'].mean():.3f}  |  Plage : [{df['int_corr'].min():.2f}, {df['int_corr'].max():.2f}]")
print(f"match=0 : {df[df['match']==0]['int_corr'].mean():.3f}  |  match=1 : {df[df['match']==1]['int_corr'].mean():.3f}")
print(f"=> Différence faible : les intérêts communs ont peu d'impact sur le match")
 
print(f"\nattr_expectation_gap (attractivité voulue T1 - attractivité reçue soirée) :")
print(f"Moyenne : {df['attr_expectation_gap'].mean():.2f}  (négatif = reçoit MOINS que ce qu'il voulait)")
print(f"Médiane : {df['attr_expectation_gap'].median():.2f}")
print(f"=> {(df['attr_expectation_gap'] < 0).mean()*100:.0f}% des participants reçoivent moins d'attractivité qu'espéré")
print(f"=> Les participants surestiment souvent ce qu’ils pensent mériter")
 
print(f"\nsamerace :")
print(f"match=0 : {df[df['match']==0]['samerace'].mean()*100:.1f}% même race")
print(f"match=1 : {df[df['match']==1]['samerace'].mean()*100:.1f}% même race")
print(f"=> La proximité d’origine raciale change peu entre match et non-match")
 
fig_u6 = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        "Écart d'âge absolu (age_diff)",
        "Corrélation intérêts communs (int_corr)",
        "Expectation gap attractivité",
        "Score moyen reçu (score_received_avg)"
    ],
    vertical_spacing=0.18, horizontal_spacing=0.12,
)
 
# age_diff
fig_u6.add_trace(go.Histogram(
    x=df["age_diff"], nbinsx=25,
    marker_color=C["blue"], opacity=0.85, showlegend=False,
), row=1, col=1)
fig_u6.add_vline(x=df["age_diff"].mean(), line_dash="dash", line_color="black",
    annotation_text=f"μ={df['age_diff'].mean():.1f}", row=1, col=1)
 
# int_corr
fig_u6.add_trace(go.Histogram(
    x=df["int_corr"], nbinsx=30,
    marker_color=C["teal"], opacity=0.85, showlegend=False,
), row=1, col=2)
fig_u6.add_vline(x=0, line_dash="dot", line_color=C["neutral"],
    annotation_text="0", row=1, col=2)
 
# attr_expectation_gap
fig_u6.add_trace(go.Histogram(
    x=df["attr_expectation_gap"], nbinsx=30,
    marker_color=C["coral"], opacity=0.85, showlegend=False,
), row=2, col=1)
fig_u6.add_vline(x=0, line_dash="dot", line_color="black",
    annotation_text="0 (équilibre)", row=2, col=1)
fig_u6.add_vline(x=df["attr_expectation_gap"].mean(), line_dash="dash",
    line_color=C["no_match"],
    annotation_text=f"μ={df['attr_expectation_gap'].mean():.2f}", row=2, col=1)
 
# score_received_avg
for genre, clr in [("Femme", C["female"]), ("Homme", C["male"])]:
    sub = df[df["gender_label"] == genre]
    fig_u6.add_trace(go.Histogram(
        x=sub["score_received_avg"], nbinsx=25,
        name=genre, marker_color=clr, opacity=0.65,
    ), row=2, col=2)
 
fig_u6.update_layout(
    title="Écart d’âge, surprises sur l’attractivité et autres signaux agrégés",
    height=560, plot_bgcolor="white", title_font_size=15,
    margin=dict(t=95, b=75, l=75, r=45),
    barmode="overlay", legend_title="Genre",
)
fig_u6.update_xaxes(title_text="Écart d'âge (années)", row=1, col=1)
fig_u6.update_xaxes(title_text="Corrélation (-1 à +1)", row=1, col=2)
fig_u6.update_xaxes(title_text="Gap attractivité (0-10)", row=2, col=1)
fig_u6.update_xaxes(title_text="Score moyen reçu (0-10)", row=2, col=2)
fig_u6.show()


─────────────────────────────────────────────────────────────────
Âge relatif, attentes vs réalité, signaux agrégés
─────────────────────────────────────────────────────────────────

age_diff (écart d'âge absolu) :
Moyenne : 3.65 ans  |  Médiane : 3.0 ans
75% des paires ont moins de 5 ans d'écart

int_corr (corrélation des intérêts communs) :
Moyenne : 0.195  |  Plage : [-0.83, 0.91]
match=0 : 0.191  |  match=1 : 0.217
=> Différence faible : les intérêts communs ont peu d'impact sur le match

attr_expectation_gap (attractivité voulue T1 - attractivité reçue soirée) :
Moyenne : -3.94  (négatif = reçoit MOINS que ce qu'il voulait)
Médiane : -4.00
=> 93% des participants reçoivent moins d'attractivité qu'espéré
=> Les participants surestiment souvent ce qu’ils pensent mériter

samerace :
match=0 : 39.5% même race
match=1 : 41.4% même race
=> La proximité d’origine raciale change peu entre match et non-match


### **3. Analyse bivariée — quand deux variables racontent la même histoire**

Ici, on croise les notes avec le résultat (match ou non), les genres, les préférences déclarées avec le comportement réel, et les liens statistiques entre indicateurs.


In [10]:
# =============================================================================
# NOTES DE LA SOIRÉE SELON QU’IL Y A MATCH OU NON
# =============================================================================
print("\n" + "─" * 65)
print("Notes selon le résultat de la rencontre")
print("─" * 65)
 
deltas = {}
print(f"\n  {'Attribut':<22} {'match=0':>8} {'match=1':>8} {'Delta':>8}  Signal")
print("  " + "─" * 58)
for col, fr in zip(ATTR6, ATTR6_FR):
    m0 = df[df["match"]==0][col].mean()
    m1 = df[df["match"]==1][col].mean()
    d  = m1 - m0
    deltas[col] = d
    signal = "Fort"  if d > 1.2 else "Modéré" if d > 0.7 else "Faible"
    print(f"  {fr:<22} {m0:>8.3f} {m1:>8.3f} {d:>+8.3f}  {signal}")
 
print(f"\nÉcarts les plus marqués : Fun (Delta={deltas['fun']:+.3f}), "
      f"Shar (Delta={deltas['shar']:+.3f}), Attr (Delta={deltas['attr']:+.3f})")
print(f"- Sinc et Intel : forts en absolu (~7) mais peu discriminants (Delta<0.8)")
print(f"- L'effet politesse gonfle les notes de sincérité/intelligence")
 
# Boxplots
fig_b1 = make_subplots(
    rows=2, cols=3,
    subplot_titles=ATTR6_FR,
    vertical_spacing=0.16, horizontal_spacing=0.08,
)
for i, col in enumerate(ATTR6):
    r, c = divmod(i, 3)
    for match_val, clr, name in [(0, C["no_match"], "Pas de match"), (1, C["match"], "Match")]:
        fig_b1.add_trace(go.Box(
            y=df[df["match"]==match_val][col],
            name=name,
            marker_color=clr,
            boxmean=True,
            showlegend=(i == 0),
            legendgroup=name,
        ), row=r+1, col=c+1)
 
fig_b1.update_layout(
    title="Répartition des six dimensions notées quand il y a match ou non",
    height=560, plot_bgcolor="white", title_font_size=15,
    margin=dict(t=95, b=75, l=75, r=45),
    boxmode="group", legend_title="Résultat",
)
fig_b1.update_yaxes(title_text="Score (0-10)")
fig_b1.show()
 
# Delta chart
fig_b1b = go.Figure()
sorted_attrs = sorted(zip(ATTR6_FR, [deltas[c] for c in ATTR6]),
                      key=lambda x: x[1], reverse=True)
labels_s, deltas_s = zip(*sorted_attrs)
colors_delta = [C["match"] if d > 1.0 else C["amber"] if d > 0.7
                else C["neutral"] for d in deltas_s]
fig_b1b.add_trace(go.Bar(
    x=list(labels_s), y=list(deltas_s),
    marker_color=colors_delta,
    text=[f"+{d:.3f}" for d in deltas_s],
    textposition="outside", cliponaxis=False,
))
fig_b1b.add_hline(y=1.0, line_dash="dot", line_color=C["match"],
    annotation_text="Repère : écart marqué (1 point)")
fig_b1b.add_hline(y=0.7, line_dash="dot", line_color=C["amber"],
    annotation_text="Repère : écart modéré (0,7 point)")
fig_b1b.update_layout(
    title="Écart moyen de score entre rencontres avec match et sans match",
    height=400, plot_bgcolor="white", title_font_size=15,
    margin=dict(t=95, b=75, l=75, r=45),
    yaxis_title="Écart moyen de note (avec match − sans)", showlegend=False,
)
fig_b1b.show()



─────────────────────────────────────────────────────────────────
Notes selon le résultat de la rencontre
─────────────────────────────────────────────────────────────────

  Attribut                match=0  match=1    Delta  Signal
  ──────────────────────────────────────────────────────────
  Attractivité              5.974    7.305   +1.331  Fort
  Sincérité                 7.055    7.802   +0.747  Modéré
  Intelligence              7.242    7.939   +0.697  Faible
  Fun                       6.182    7.583   +1.401  Fort
  Ambition                  6.688    7.310   +0.622  Faible
  Intérêts communs          5.266    6.644   +1.378  Fort

Écarts les plus marqués : Fun (Delta=+1.401), Shar (Delta=+1.378), Attr (Delta=+1.331)
- Sinc et Intel : forts en absolu (~7) mais peu discriminants (Delta<0.8)
- L'effet politesse gonfle les notes de sincérité/intelligence


In [11]:
# =============================================================================
# FEMMES ET HOMMES NE RÉAGISSENT PAS TOUJOURS PAREIL
# =============================================================================
print("\n" + "─" * 65)
print("Femmes et hommes : où les écarts changent")
print("─" * 65)
 
print(f"\n  Écart de score (avec match − sans match), par genre et attribut :")
print(f"  {'Attribut':<22} {'Femme':>10} {'Homme':>10}  Écart le plus fort chez")
print("  " + "─" * 60)
 
gender_gaps = {}
for col, fr in zip(ATTR6[:5], ATTR6_FR[:5]):  # 5 attrs principaux
    df_f = df[df["gender"]==0]
    df_m = df[df["gender"]==1]
    d_f = df_f[df_f["match"]==1][col].mean() - df_f[df_f["match"]==0][col].mean()
    d_m = df_m[df_m["match"]==1][col].mean() - df_m[df_m["match"]==0][col].mean()
    gender_gaps[col] = (d_f, d_m)
    winner = "Femme" if d_f > d_m + 0.1 else "Homme" if d_m > d_f + 0.1 else "= Égal"
    print(f"  {fr:<22} {d_f:>+10.3f} {d_m:>+10.3f}  {winner}")
 
print(f"\nFun : les femmes y sont plus sensibles (+{gender_gaps['fun'][0]:.2f} vs +{gender_gaps['fun'][1]:.2f})")
print(f"Attr : légèrement plus fort pour les femmes ({gender_gaps['attr'][0]:+.2f} vs {gender_gaps['attr'][1]:+.2f})")
print(f"Sinc & Intel : discrimination faible pour les deux genres")
 
# Taux de décision par genre
dec_f = df[df["gender"]==0]["dec"].mean()*100
dec_m = df[df["gender"]==1]["dec"].mean()*100
print(f"\nTaux décision 'Oui' — Femme: {dec_f:.1f}%  Homme: {dec_m:.1f}%")
print(f"=> Les hommes sont {dec_m/dec_f:.1f}x plus permissifs dans leurs décisions")
 
fig_b2 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "Sensibilité aux attributs selon le genre et le résultat",
        "Distribution attr par genre x match"
    ],
    horizontal_spacing=0.12,
)
 
# Grouped bar : gap par genre
attrs5    = ATTR6_FR[:5]
gaps_f  = [gender_gaps[c][0] for c in ATTR6[:5]]
gaps_m  = [gender_gaps[c][1] for c in ATTR6[:5]]
fig_b2.add_trace(go.Bar(
    x=attrs5, y=gaps_f, name="Femme",
    marker_color=C["female"], opacity=0.85,
), row=1, col=1)
fig_b2.add_trace(go.Bar(
    x=attrs5, y=gaps_m, name="Homme",
    marker_color=C["male"], opacity=0.85,
), row=1, col=1)
 
# Violin attr par genre × match
for genre, gval, clr_use in [("Femme", 0, C["female"]), ("Homme", 1, C["male"])]:
    for match_val, opacity, suffix in [(0, 0.4, "— Pas de match"), (1, 0.9, "— Match")]:
        sub = df[(df["gender"]==gval) & (df["match"]==match_val)]
        fig_b2.add_trace(go.Box(
            y=sub["attr"],
            name=f"{genre} {suffix}",
            marker_color=clr_use,
            opacity=opacity,
            boxmean=True,
            
            showlegend=False,
        ), row=1, col=2)
 
fig_b2.update_layout(
    title="Sensibilité aux attributs selon le genre",
    height=440, plot_bgcolor="white", title_font_size=15,
    margin=dict(t=95, b=75, l=75, r=45),
    barmode="group", legend_title="Genre",
)
fig_b2.update_yaxes(title_text="Écart moyen de score (avec match − sans)", row=1, col=1)
fig_b2.update_yaxes(title_text="Score attractivité (0-10)", row=1, col=2)
fig_b2.show()



─────────────────────────────────────────────────────────────────
Femmes et hommes : où les écarts changent
─────────────────────────────────────────────────────────────────

  Écart de score (avec match − sans match), par genre et attribut :
  Attribut                    Femme      Homme  Écart le plus fort chez
  ────────────────────────────────────────────────────────────
  Attractivité               +1.405     +1.260  Femme
  Sincérité                  +0.854     +0.643  Femme
  Intelligence               +0.752     +0.642  Femme
  Fun                        +1.600     +1.204  Femme
  Ambition                   +0.586     +0.657  = Égal

Fun : les femmes y sont plus sensibles (+1.60 vs +1.20)
Attr : légèrement plus fort pour les femmes (+1.40 vs +1.26)
Sinc & Intel : discrimination faible pour les deux genres

Taux décision 'Oui' — Femme: 36.8%  Homme: 47.4%
=> Les hommes sont 1.3x plus permissifs dans leurs décisions


In [12]:
# =============================================================================
# CE QU’ON DIT VERSUS CE QU’ON NOTE ENSUITE
# =============================================================================
print("\n" + "─" * 65)
print("Priorités annoncées et notes réelles le soir même")
print("─" * 65)
 
pairs = [
    ("attr1_1","attr","Attractivité"),
    ("sinc1_1","sinc","Sincérité"),
    ("intel1_1","intel","Intelligence"),
    ("fun1_1","fun","Fun"),
    ("amb1_1","amb","Ambition"),
]
 
print(f"\nCe que je dis vouloir (T1) vs ce que je note vraiment (soirée) :")
print(f"{'Attribut':<22} {'Déclaré F':>10} {'Réel F':>10} {'Déclaré H':>10} {'Réel H':>10}")
print("  " + "─" * 66)
 
decl_f, reel_f, decl_m, reel_m, attrs_label = [], [], [], [], []
for p, s, fr in pairs:
    df_f = df[df["gender"]==0]
    df_m = df[df["gender"]==1]
    vdf = df_f[p].mean(); vrf = df_f[s].mean()
    vdm = df_m[p].mean(); vrm = df_m[s].mean()
    decl_f.append(vdf); reel_f.append(vrf)
    decl_m.append(vdm); reel_m.append(vrm)
    attrs_label.append(fr)
    print(f"{fr:<22} {vdf:>10.2f} {vrf:>10.2f} {vdm:>10.2f} {vrm:>10.2f}")
 
print(f"\nObservations clés :")
print(f"->Les scores 'réels' sont bien plus hauts que les préférences déclarées")
print(f"(effet : l'échelle 100pts génère des valeurs compressées en T1)")
print(f"->Les hommes déclarent plus d'importance à l'attractivité (2.69 vs 1.82)")
print(f"->Dans les notes réelles soirée : les femmes notent l'attr plus sévèrement")
print(f"->La sincérité/intelligence sont sur-déclarées, peu discriminantes en réel")
 
# Corrélation pref déclarée vs match (est-ce que ce que je veux prédit le match ?)
print(f"\n  Corrélation préférences T1 avec match :")
for p, s, fr in pairs:
    corr_p = df[p].corr(df["match"])
    corr_s = df[s].corr(df["match"])
    print(f"  {fr:<22}  corr({p},match)={corr_p:+.3f}  corr({s},match)={corr_s:+.3f}")
print(f"->Les préférences déclarées T1 corrèlent quasi 0 avec le match")
print(f"->Les scores soirée réels corrèlent fortement avec le match")
print(f"->Conclusion : ce qu'on dit vouloir ≠ ce qui crée un match")
 
fig_b3 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        "Déclaré T1 vs Score réel soirée — Femme",
        "Déclaré T1 vs Score réel soirée — Homme"
    ],
    horizontal_spacing=0.14,
)
x_labels = [fr for _, _, fr in pairs]
for col_idx, (gval, genre, d_vals, r_vals) in enumerate([
    (0, "Femme", decl_f, reel_f),
    (1, "Homme", decl_m, reel_m),
]):
    clr = C["female"] if gval==0 else C["male"]
    fig_b3.add_trace(go.Bar(
        x=x_labels, y=d_vals, name="Déclaré T1",
        marker_color=clr, opacity=0.5,
        showlegend=(col_idx==0),
    ), row=1, col=col_idx+1)
    fig_b3.add_trace(go.Bar(
        x=x_labels, y=r_vals, name="Note réelle soirée",
        marker_color=clr, opacity=0.95,
        showlegend=(col_idx==0),
    ), row=1, col=col_idx+1)
 
fig_b3.update_layout(
    title="Ce que je dis vouloir (T1) vs ce que je note vraiment (soirée)",
    height=420, plot_bgcolor="white", title_font_size=15,
    margin=dict(t=95, b=75, l=75, r=45),
    barmode="group", legend_title="Type",
)
fig_b3.update_yaxes(title_text="Score moyen (0-10)")
fig_b3.show()


─────────────────────────────────────────────────────────────────
Priorités annoncées et notes réelles le soir même
─────────────────────────────────────────────────────────────────

Ce que je dis vouloir (T1) vs ce que je note vraiment (soirée) :
Attribut                Déclaré F     Réel F  Déclaré H     Réel H
  ──────────────────────────────────────────────────────────────────
Attractivité                 1.81       5.93       2.69       6.45
Sincérité                    1.83       7.11       1.65       7.25
Intelligence                 2.10       7.43       1.95       7.28
Fun                          1.71       6.30       1.78       6.52
Ambition                     1.29       6.97       0.86       6.61

Observations clés :
->Les scores 'réels' sont bien plus hauts que les préférences déclarées
(effet : l'échelle 100pts génère des valeurs compressées en T1)
->Les hommes déclarent plus d'importance à l'attractivité (2.69 vs 1.82)
->Dans les notes réelles soirée : les femmes noten

In [13]:
# =============================================================================
# LIENS ENTRE LES INDICATEURS
# =============================================================================
print("\n" + "─" * 65)
print("Carte des liens statistiques entre variables")
print("─" * 65)
 
corr_cols = (ATTR6 + ["like","prob"] +
             ["attr_o","fun_o","like_o","prob_o","shar_o"] +
             PREF_COLS[:4] +
             ["int_corr","samerace","age_diff","attr_expectation_gap","match"])
corr_cols = [c for c in corr_cols if c in df.columns]
corr_matrix = df[corr_cols].corr().round(3)
 
# Labels lisibles
label_map = {
    "attr":"attr↓","sinc":"sinc↓","intel":"intel↓","fun":"fun↓",
    "amb":"amb↓","shar":"shar↓","like":"like↓","prob":"prob↓",
    "attr_o":"attr_o↑","fun_o":"fun_o↑","like_o":"like_o↑",
    "prob_o":"prob_o↑","shar_o":"shar_o↑",
    "attr1_1":"attr_T1","sinc1_1":"sinc_T1","intel1_1":"intel_T1","fun1_1":"fun_T1",
    "int_corr":"int_corr","samerace":"samerace","age_diff":"age_diff",
    "attr_expectation_gap":"attr_gap","match":"MATCH",
}
labels = [label_map.get(c, c) for c in corr_cols]
 
fig_b4 = go.Figure(go.Heatmap(
    z=corr_matrix.values,
    x=labels, y=labels,
    colorscale="RdBu",
    zmid=0, zmin=-1, zmax=1,
    text=corr_matrix.values.round(2),
    texttemplate="%{text}",
    textfont=dict(size=9),
    colorbar=dict(title="Corrélation"),
))
fig_b4.update_layout(
    title="Liens entre indicateurs et probabilité de match",
    height=680, width=780,
    xaxis=dict(tickangle=-45),
    title_font_size=15,
    margin=dict(t=95, b=75, l=75, r=45),
)
fig_b4.show()


─────────────────────────────────────────────────────────────────
Carte des liens statistiques entre variables
─────────────────────────────────────────────────────────────────


In [14]:
# =============================================================================
# CONTEXTE DE LA RENCONTRE ET RÉSULTAT
# =============================================================================
print("\n" + "─" * 65)
print("Même milieu, écart d’âge, intérêts en commun…")
print("─" * 65)
 
ctx_vars = [
    ("samerace", "Même race", "catégorielle"),
    ("int_corr", "Corrélation intérêts","continue"),
    ("age_diff", "Écart d'âge absolu", "continue"),
    ("attr_expectation_gap","Gap attractivité", "continue"),
]
print(f"\n{'Variable':<30} {'match=0':>10} {'match=1':>10} {'Delta':>8}  Verdict")
print("  " + "─" * 70)

for col, fr, typ in ctx_vars:
    if col not in df.columns:
        continue
    m0 = df[df["match"]==0][col].mean()
    m1 = df[df["match"]==1][col].mean()
    d  = m1 - m0
    verdict = "Signal négligeable" if abs(d) < 0.1 else \
              "Signal faible"      if abs(d) < 0.5 else "Signal modéré"
    if col == "samerace":
        verdict = "Signal négligeable"
    print(f"  {fr:<30} {m0:>10.3f} {m1:>10.3f} {d:>+8.3f}  {verdict}")
 

fig_b5 = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        "Même race vs match (%)",
        "Proximité des centres d’intérêt selon le résultat",
        "Écart d’âge selon le résultat",
    ],
    horizontal_spacing=0.12,
)
 
# Samerace — taux de match par groupe
sr_match = df.groupby("samerace")["match"].mean().reset_index()
sr_match["label"] = sr_match["samerace"].map({0:"Races diff.", 1:"Même race"})
fig_b5.add_trace(go.Bar(
    x=sr_match["label"], y=(sr_match["match"]*100).round(1),
    marker_color=[C["neutral"], C["teal"]],
    text=(sr_match["match"]*100).round(1).astype(str)+"%",
    textposition="outside", cliponaxis=False, showlegend=False,
), row=1, col=1)
fig_b5.add_hline(y=df["match"].mean()*100, line_dash="dot",
    line_color=C["neutral"], row=1, col=1)
 
# int_corr boxplot par match
for mv, clr, nm in [(0,C["no_match"],"Pas de match"),(1,C["match"],"Match")]:
    fig_b5.add_trace(go.Box(
        y=df[df["match"]==mv]["int_corr"],
        name=nm, marker_color=clr,
        boxmean=True, showlegend=False,
    ), row=1, col=2)
 
# age_diff boxplot par match
for mv, clr, nm in [(0,C["no_match"],"Pas de match"),(1,C["match"],"Match")]:
    fig_b5.add_trace(go.Box(
        y=df[df["match"]==mv]["age_diff"],
        name=nm, marker_color=clr,
        boxmean=True, showlegend=False,
    ), row=1, col=3)
 
fig_b5.update_layout(
    title="Contexte des rencontres et part de match",
    height=420, plot_bgcolor="white", title_font_size=15,
    margin=dict(t=95, b=75, l=75, r=45),
    boxmode="group",
)
fig_b5.update_yaxes(title_text="% match", row=1, col=1)
fig_b5.update_yaxes(title_text="int_corr", row=1, col=2)
fig_b5.update_yaxes(title_text="Écart d'âge (années)", row=1, col=3)
fig_b5.show()


─────────────────────────────────────────────────────────────────
Même milieu, écart d’âge, intérêts en commun…
─────────────────────────────────────────────────────────────────

Variable                          match=0    match=1    Delta  Verdict
  ──────────────────────────────────────────────────────────────────────
  Même race                           0.395      0.414   +0.019  Signal négligeable
  Corrélation intérêts                0.191      0.217   +0.026  Signal négligeable
  Écart d'âge absolu                  3.739      3.175   -0.563  Signal modéré
  Gap attractivité                   -3.725     -5.019   -1.294  Signal modéré


In [16]:
# =============================================================================
# APPRÉCIATION GLOBALE ET INTUITION DE MATCH
# =============================================================================
print("\n" + "─" * 65)
print("Les notes « j’aimerais revoir » et « je crois au match »")
print("─" * 65)
 
print(f"\nCorrélation avec match :")
print(f"like  : {df['like'].corr(df['match']):+.3f}")
print(f"prob  : {df['prob'].corr(df['match']):+.3f}")
print(f"like_o: {df['like_o'].corr(df['match']):+.3f}")
print(f"attr  : {df['attr'].corr(df['match']):+.3f}")
 
print(f"\nTaux de match par seuil de like :")
thresholds = [4, 5, 6, 7, 8, 9]
like_rates, prob_rates = [], []
for t in thresholds:
    rl = df[df["like"]>=t]["match"].mean()*100
    rp = df[df["prob"]>=t]["match"].mean()*100
    like_rates.append(rl)
    prob_rates.append(rp)
    print(f"like>={t}: {rl:.1f}%   prob>={t}: {rp:.1f}%")
 
print(f"\nQuelqu'un noté like>=8 a {df[df['like']>=8]['match'].mean()*100:.0f}% de chance de matcher")
print(f"vs taux global de {df['match'].mean()*100:.1f}% — soit {df[df['like']>=8]['match'].mean()/df['match'].mean():.1f}x plus élevé")
print(f"like et prob reflètent l'intuition globale — ils capturent tout en un chiffre")
print(f"Une note composite résumerait bien l’intuition de succès")
 
fig_b6 = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        "Part de match selon le seuil choisi (like ou prod)",
        "Note « j’aimerais revoir » selon le résultat",
        "Quand les deux appréciations se croisent"
    ],
    horizontal_spacing=0.11,
)
 
# Courbe taux match vs seuil
fig_b6.add_trace(go.Scatter(
    x=thresholds, y=like_rates,
    mode="lines+markers", name="like ≥ seuil",
    marker=dict(color=C["match"], size=9),
    line=dict(color=C["match"], width=2.5),
), row=1, col=1)
fig_b6.add_trace(go.Scatter(
    x=thresholds, y=prob_rates,
    mode="lines+markers", name="prob ≥ seuil",
    marker=dict(color=C["purple"], size=9),
    line=dict(color=C["purple"], width=2.5),
), row=1, col=1)
fig_b6.add_hline(y=df["match"].mean()*100, line_dash="dot",
    line_color=C["neutral"],
    annotation_text=f"Base : {df['match'].mean()*100:.1f}%",
    row=1, col=1)
 
# Violin like par match
for mv, clr, nm in [(0,C["no_match"],"Pas de match"),(1,C["match"],"Match")]:
    fig_b6.add_trace(go.Box(
        y=df[df["match"]==mv]["like"],
        name=nm, marker_color=clr,
        boxmean=True, 
        showlegend=False, opacity=0.8,
    ), row=1, col=2)
 
# Scatter like_o vs like coloré par match
sample = df.sample(2000, random_state=42)
fig_b6.add_trace(go.Scatter(
    x=sample["like"], y=sample["like_o"],
    mode="markers",
    marker=dict(
        color=sample["match"].map({0:C["no_match"], 1:C["match"]}),
        size=4, opacity=0.5,
    ),
    showlegend=False,
), row=1, col=3)
 
fig_b6.update_layout(
    title="Appréciation globale, intuition de match et lien avec le résultat",
    height=420, plot_bgcolor="white", title_font_size=15,
    margin=dict(t=95, b=75, l=75, r=45),
    legend_title="Variable",
)
fig_b6.update_xaxes(title_text="Seuil minimum", row=1, col=1)
fig_b6.update_yaxes(title_text="% match", row=1, col=1)
fig_b6.update_yaxes(title_text="Score like (0-10)", row=1, col=2)
fig_b6.update_xaxes(title_text="like (sujet -> partenaire)", row=1, col=3)
fig_b6.update_yaxes(title_text="like_o (partenaire -> sujet)", row=1, col=3)
fig_b6.show()


─────────────────────────────────────────────────────────────────
Les notes « j’aimerais revoir » et « je crois au match »
─────────────────────────────────────────────────────────────────

Corrélation avec match :
like  : +0.303
prob  : +0.252
like_o: +0.303
attr  : +0.256

Taux de match par seuil de like :
like>=4: 17.9%   prob>=4: 19.3%
like>=5: 19.4%   prob>=5: 21.1%
like>=6: 22.5%   prob>=6: 24.7%
like>=7: 28.2%   prob>=7: 30.0%
like>=8: 35.1%   prob>=8: 36.5%
like>=9: 39.0%   prob>=9: 43.5%

Quelqu'un noté like>=8 a 35% de chance de matcher
vs taux global de 16.4% — soit 2.1x plus élevé
like et prob reflètent l'intuition globale — ils capturent tout en un chiffre
Une note composite résumerait bien l’intuition de succès


### **Segments, notes et intuition de succès**

Sens du rendez-vous (qui rencontre qui), attractivité vue par tranches, proximité ou écart entre deux personnes, priorités déclarées avant la soirée, et ce que la probabilité perçue dit du résultat réel.


In [17]:
# =============================================================================
# Segments, tranches d’attractivité et synthèses
# =============================================================================
# Libellés de sens de la rencontre
if "segment_genre" not in df.columns and "gender" in df.columns:
    df["segment_genre"] = df["gender"].map({1: "Homme -> Femme", 0: "Femme -> Homme"})

# A) Taux de match par segment
if {"segment_genre", "match"}.issubset(df.columns):
    seg = (df.groupby("segment_genre")["match"].mean() * 100).reset_index(name="Taux match (%)")
    fig_seg = px.bar(seg, x="segment_genre", y="Taux match (%)", color="segment_genre",
                     title="Taux de match selon le sens de la rencontre",
                     color_discrete_map={"Homme -> Femme": "#378ADD", "Femme -> Homme": "#D4537E"})
    fig_seg.update_layout(showlegend=False, height=420)
    fig_seg.show()

# B) Match vs attractivité (vue simple par tranches)
if {"attr", "match"}.issubset(df.columns):
    tmp = df[["attr", "match"]].dropna().copy()
    tmp["attr_bin"] = pd.cut(tmp["attr"], bins=[0, 2, 4, 6, 8, 10],
                             labels=["0-2", "2-4", "4-6", "6-8", "8-10"], include_lowest=True)
    ma = (tmp.groupby("attr_bin", observed=False)["match"].mean() * 100).reset_index(name="Taux match (%)")
    fig_ma = px.bar(ma, x="attr_bin", y="Taux match (%)", title="Match selon la note d’attractivité (par tranches)")
    fig_ma.update_layout(height=420)
    fig_ma.show()

# C) Écarts entre partenaires (alignement)
if {"attr", "attr_o", "like", "like_o", "match"}.issubset(df.columns):
    df["gap_attr"] = (df["attr"] - df["attr_o"]).abs()
    df["gap_like"] = (df["like"] - df["like_o"]).abs()
    gs = df.groupby("match")[["gap_attr", "gap_like"]].mean().reset_index()
    gs["match_label"] = gs["match"].map({0: "Pas de match", 1: "Match"})
    gs_l = gs.melt(id_vars=["match", "match_label"], var_name="Type", value_name="Écart moyen")
    gs_l["Type"] = gs_l["Type"].map({"gap_attr": "Écart attractivité", "gap_like": "Écart intérêt"})
    fig_gap = px.bar(gs_l, x="Type", y="Écart moyen", color="match_label", barmode="group",
                     title="Alignement perçu : écarts moyens",
                     color_discrete_map={"Match": "#1D9E75", "Pas de match": "#E24B4A"})
    fig_gap.update_layout(height=420)
    fig_gap.show()

# Synthèses : priorités déclarées, intérêts, confiance
pairs = [
    ("attr1_1", "attr", "Attractivité"),
    ("sinc1_1", "sinc", "Sincérité"),
    ("intel1_1", "intel", "Intelligence"),
    ("fun1_1", "fun", "Fun"),
    ("amb1_1", "amb", "Ambition"),
    ("shar1_1", "shar", "Intérêts communs"),
]
pref_cols = [p for p, _, _ in pairs if p in df.columns]
if pref_cols and "gender_label" in df.columns:
    pref_g = df.groupby("gender_label")[pref_cols].mean().T.reset_index().rename(columns={"index": "pref_col"})
    pref_g["Attribut"] = pref_g["pref_col"].map(dict((p, l) for p, _, l in pairs))
    pl = pref_g.melt(id_vars=["pref_col", "Attribut"], var_name="Genre", value_name="Importance")
    fig_q1 = px.bar(pl, x="Attribut", y="Importance", color="Genre", barmode="group",
                    title="Priorités déclarées avant la soirée (score bas = peu prioritaire)",
                    color_discrete_map={"Femme": "#D4537E", "Homme": "#378ADD"})
    fig_q1.update_layout(height=480)
    fig_q1.show()

if {"int_corr", "samerace", "match"}.issubset(df.columns):
    q3 = pd.DataFrame({
        "Variable": ["Intérêts communs (int_corr)", "Même race (samerace)"],
        "Corr. avec match": [df["int_corr"].corr(df["match"]), df["samerace"].corr(df["match"])],
    })
    fig_q3 = px.bar(q3, x="Variable", y="Corr. avec match", title="Lien statistique avec le match : intérêts en commun et même origine")
    fig_q3.update_layout(showlegend=False, height=400)
    fig_q3.show()

if {"prob", "match"}.issubset(df.columns):
    cal = df[["prob", "match"]].dropna().copy()
    cal["prob_bin"] = pd.cut(cal["prob"], bins=[0, 2, 4, 6, 8, 10], include_lowest=True)
    cg = cal.groupby("prob_bin", observed=False).agg(prob_m=("prob", "mean"), m=("match", "mean")).reset_index()
    cg["Taux match (%)"] = (cg["m"] * 100).round(1)
    fig_q4 = px.line(cg, x="prob_m", y="Taux match (%)", markers=True, title="Confiance dans un match : ce qu’on croit et ce qui se passe")
    fig_q4.update_layout(height=420)
    fig_q4.show()

order_cols = [c for c in df.columns if any(k in c.lower() for k in ["order", "position", "first", "last", "round"])]
print("\nVariables d’ordre des rendez-vous (premier / dernier) :", order_cols)
if not order_cols:
    print("Pas de colonne d’ordre exploitable dans ce jeu de données.")



Variables d’ordre des rendez-vous (premier / dernier) : []
Pas de colonne d’ordre exploitable dans ce jeu de données.


### **À retenir pour la suite**

- **Sens du rendez-vous** : quand les taux diffèrent selon qui note qui, il vaut mieux distinguer les situations plutôt que tout mélanger dans une seule lecture.
- **Attractivité perçue** : la part de match augmente en général avec la note reçue — un repère clair pour ce qui attire l’attention en première impression.
- **Proximité des notes** : lorsque les évaluations se rapprochent entre les deux personnes, l’expérience paraît plus cohérente — un fil conducteur pour la suite de l’analyse.
- **Alignement déclaré / vécu** : comparer ce qu’on dit prioriser avant la soirée et ce qu’on note ensuite évite les lectures trop littérales des questionnaires.
- **Confiance dans un match** : comparer ce qu’on croit avant le verdict et ce qui arrive vraiment évite les surinterprétations du « feeling ».
